# TL;DR. 
Below we instantiate and run inference on Mistral LLMs, mainly using the `transformers` HF library.  

Other client APIs like OpenAI Python SDK or the OpenAI REST API are included for models,  

and we can use then as long as a local vLLM web server is up and running  

__NOTE!__ The vLLM server for `mistralai/Magistral-small-2506` does not work.

In [ ]:
import os

# Use if for gated LLMs
from huggingface_hub import login
login(os.environ["fine-grained-token"])

In [11]:
from typing import List
import os, re

# The default pattern used below splits sentences.
# We do not use capturing parentheses so that split characters are excluded from the result
# https://docs.python.org/3.12/library/re.html#re.split

# To test if a split pattern works, compare with the original strin using capturing parentheses, e.g.
# SYSTEM_PROMPT == ''.join(re.split(f"({splitpat})", SYSTEM_PROMPT))

def pattern_split(
    text: str, 
    patternlist: List[re.Pattern]=[re.compile(r"(?<=[.:;!?])\s+")],
    joinstr: str=os.linesep
) -> str:
    ''' Split a (long) string into multiple lines 
        (or do sth more general using multiple split patterns and 
        join string).
        I use this below to split LLM outputs into short lines.
    '''
    inputs, outputs = [text], []
    for pat in patternlist:
        _ = [outputs.extend(pat.split(s)) for s in inputs]
        inputs, outputs = outputs, []
    return joinstr.join(inputs)

# Testing
# print(pattern_split(text="One. Two; and three"))

In [29]:
from typing import List

from pydantic import BaseModel, SkipValidation
from functools import cache
from transformers import AutoTokenizer, AutoModelForCausalLM

# Refrain from calling these inside the notebook if cache is not available. 
# Takes too long. Use the console instead
from contextlib import redirect_stdout
import io

class ModelConfig(BaseModel):
    ''' Use this model to instantiate and run inference easily:
        model = ModelConfig.from_pretrained(<model-name>)
        model.generate(<query>)
    '''

    # This is required to set the member typehints to Auto*
    model_config = dict(arbitrary_types_allowed=True)

    model_id: str
    tokenizer: SkipValidation[AutoTokenizer]
    model: SkipValidation[AutoModelForCausalLM]
    
    # The order of decorators does matter
    @classmethod
    @cache
    def from_pretrained(cls, model_id: str):
        with redirect_stdout(io.StringIO()):
            return ModelConfig(            
                model_id=model_id,
                tokenizer=AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id),
                model=AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id)
            )
    
    def generate(self, prompt_ids: List[int], max_new_tokens=512, temperature=.7, top_p=.9) -> str:
        # inputs = self.tokenizer(query, return_tensors="pt")
        token_ids = self.model.generate(
            prompt_ids, do_sample=True, 
            max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p
        )
        return self.tokenizer.batch_decode(
            token_ids, 
            skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

# `mistral-7b-instruct-v0.1`

## Inference with `transformers.AutoModelForCausalLM`

The model card is [here](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1).  

This is a __gated__ model (meaning it requires access request and approval be an HF registered user).  

The model card uses Mistral's Python library `mistral_common` for tokenization. This is NOT mandatory however since a `LlamaTokenizerFast` can be instantiated with
```python
AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
```

So, in the cells below I use my `ModelConfig` model to instantiate the LLM from cache.  

Note that the `tokenizer_config.json` file in the HF repo (in `~/.cache/huggingface/hub/model--mistralai--Mistral-7B-Instruct-v0.1/snapshots/<revision>/tokenizer_config.json`). 

contains at its end a `jinja2` fragment with the tokenizer's `chat_template`. It is used for converting the `List[Dict[role, content]]` chat messages to a string annotated with Mistral's special tokens.  



In [16]:
model = ModelConfig.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
messages = [
    {"role": "user", "content": "What is your favourite condiment?"},
    {"role": "assistant", "content": "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!"},
    {"role": "user", "content": "Do you have mayonnaise recipes?"}
]

## Mistral Tokenization
Below we hack (at last) the Mistral message tokenization.  

This is important as it permits to write custom converters for chat inputs that do not comply with the OpenAI chat format.

In [31]:
import os
from transformers.models.llama.tokenization_llama_fast import LlamaTokenizerFast

# apply_chat_template will use the chat_template of the model, in 
# ~/.cache/huggingface/hub/model--mistralai--Mistral-7B-Instruct-v0.1/snapshots/<commit-hash>/tokenizer_config.json (scroll to the bottom to see th jinja configuration),
# to convert the OpenAI chat format to a mere string.
annotated_ids = model.tokenizer.apply_chat_template(messages, return_tensors="pt")

# annotated_ids is a 1xN Tensor of token ids of the final string. How do we obtain the string?
# The tokenizer of the ModelConfig object is a LlamaTokenizerFast
print("model.tokenizer type is LlamaTokenizerFast:", isinstance(model.tokenizer, LlamaTokenizerFast))

# ...we can therefore decode the real prompt to the LLM
print(model.tokenizer.decode(annotated_ids[0]))

model.tokenizer type is LlamaTokenizerFast: True
<s> [INST] What is your favourite condiment? [/INST] Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!</s> [INST] Do you have mayonnaise recipes? [/INST]


In [ ]:
model.generate(prompt_ids=annotated_ids, max_new_tokens=1000)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


["[INST] What is your favourite condiment? [/INST] Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen! [INST] Do you have mayonnaise recipes? [/INST] Certainly! Here's a simple recipe for mayonnaise that you can easily whip up at home:\n\nIngredients:\n\n* 2 large egg yolks\n* 1 tablespoon Dijon mustard\n* 1 tablespoon white wine vinegar or lemon juice\n* 1/2 teaspoon salt\n* 1/4 teaspoon sugar\n* 3/4 cup vegetable oil or light olive oil\n\nInstructions:\n\n1. In a medium-sized mixing bowl, whisk together the egg yolks, Dijon mustard, white wine vinegar or lemon juice, salt, and sugar until well combined.\n2. Slowly pour the vegetable oil or light olive oil into the mixture, whisking continuously as you go.\n3. Continue whisking until the mixture thickens and the oil has been fully incorporated.\n4. Taste the mayonnaise and adjust the seasoning if necessary.\n5. Cover the bowl with pla

# `magistral-small-2506`

1. According to the [model card](https://huggingface.co/mistralai/Magistral-Small-2506) we can run inference only using a `vLLM` server.  

   I could not make the model work on my Mac however! The vLLM server, started with the parameters proposed in the model card

   consumes too much memory and hangs when a request is sent by the OpenAI SDK client below.

2. There's no support for transformers (not mentioned in the model card).

## Inference on vLLM with OpenAI Py SDK

In [ ]:
# Convenience store
# 1. readonly token
rotoken = os.environ["read-only-token"]
# 2. fine-grained token
fgtoken = os.environ["fine-graoned-token"]

! huggingface-cli login --token {fgtoken}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `fine-grained-token` has been saved to /Users/christos/.cache/huggingface/stored_tokens
Your token has been saved to /Users/christos/.cache/huggingface/token
Login successful.
The current active token is: `fine-grained-token`


In [10]:
from openai import OpenAI
client = OpenAI(api_key=fgtoken, base_url="http://localhost:8000/v1")

In [11]:
# set indent to a positive integer to separate fields with newlines
print((models := client.models.list()).to_json(indent=2))
model = models.data[0].id

{
  "data": [
    {
      "id": "mistralai/Mistral-7B-Instruct-v0.1",
      "created": 1750419064,
      "object": "model",
      "owned_by": "vllm",
      "root": "mistralai/Mistral-7B-Instruct-v0.1",
      "parent": null,
      "max_model_len": 32768,
      "permission": [
        {
          "id": "modelperm-aa5fedd8c9a64d3791cbebc3bb9f004f",
          "object": "model_permission",
          "created": 1750419064,
          "allow_create_engine": false,
          "allow_sampling": true,
          "allow_logprobs": true,
          "allow_search_indices": false,
          "allow_view": true,
          "allow_fine_tuning": false,
          "organization": "*",
          "group": null,
          "is_blocking": false
        }
      ]
    }
  ],
  "object": "list"
}


In [15]:
from huggingface_hub import hf_hub_download

def load_system_prompt(repo_id: str, filename: str) -> str:
    file_path = hf_hub_download(repo_id=repo_id, filename=filename)
    with open(file_path, "r") as file:
        system_prompt = file.read()
    return system_prompt

In [ ]:
import os
import re

SYSTEM_PROMPT = load_system_prompt(model, "SYSTEM_PROMPT.txt")

# Split newlines on punct
# Without capturing parentheses the split characters are excluded from the result
# https://docs.python.org/3.12/library/re.html#re.split
print(*re.split(splitpat := r"(?<=[.:;!?])\s+", SYSTEM_PROMPT), sep=os.linesep)

# To test if the split misses any ws characters use capturing parentheses; f"({splitpat})":
SYSTEM_PROMPT == ''.join(re.split(f"({splitpat})", SYSTEM_PROMPT))

In [ ]:
query = "Write 4 sentences, each with at least 8 words. Now make absolutely sure that every sentence has exactly one word less than the previous sentence."
# or try out other queries
# query = "Exactly how many days ago did the French Revolution start? Today is June 4th, 2025."
# query = "Think about 5 random numbers. Verify if you can combine them with addition, multiplication, subtraction or division to 133"
# query = "If it takes 30 minutes to dry 12 T-shirts in the sun, how long does it take to dry 33 T-shirts?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": query}
]

# Try out non-streaming
client.chat.completions.create(
  model=model,
  messages=messages,
  stream=False,
  temperature=0.7,
  top_p=0.95,
  max_tokens=40960,
  timeout=50
)

# stream = client.chat.completions.create(
#   model=model,
#   messages=messages,
#   stream=True,
#   temperature=TEMP,
#   top_p=TOP_P,
#   max_tokens=MAX_TOK,
# )

# print("client: Start streaming chat completions...")
# printed_content = False

# for chunk in stream:
#   content = None
#   # Check the content is content
#   if hasattr(chunk.choices[0].delta, "content"):
#     content = chunk.choices[0].delta.content

#   if content is not None:
#     if not printed_content:
#         printed_content = True
#         print("\ncontent:", end="", flush=True)
#     # Extract and print the content
#     print(content, end="", flush=True)

# content:<think>
# Alright, I need to write 4 sentences where each one has at least 8 words and each subsequent sentence has one fewer word than the previous one.
# ...
# Final boxed answer (the four sentences):

# \[
# \boxed{
# \begin{aligned}
# &\text{1. The quick brown fox jumps over lazy dog and yells hello.} \\
# &\text{2. I saw the cat on the stair with my hat.} \\
# &\text{3. The man in the moon came down quickly today.} \\
# &\text{4. A cat sat on the mat today patiently.}
# \end{aligned}
# }
# \]